# 1.2 Web Search — Giving Agents Real-Time Knowledge

## What you will learn in this notebook

- Why LLMs have a **knowledge cutoff** and what that means in practice
- How to wrap **Tavily** (a search API) into a LangChain tool
- How to give an agent **live internet access** so it can answer current-events questions
- How to read the **full message trace** to see what the agent searched for

---

### The knowledge cutoff problem

Every LLM is trained on a snapshot of the internet taken at a specific date — the **training cutoff**. After that date, the model knows nothing new. Ask it about last week's election, today's stock price, or who the current mayor of a city is — and it will either guess, hallucinate, or admit it doesn't know.

```
Without web search:  "Who is the current mayor of San Francisco?"
                     → Model guesses based on stale training data  ← potentially wrong

With web search:     "Who is the current mayor of San Francisco?"
                     → Agent searches Tavily → gets today's result  ← accurate
```

### What is Tavily?

**Tavily** is a search API built specifically for LLM agents. Unlike raw Google/Bing results (which return messy HTML), Tavily returns clean, structured JSON with the most relevant excerpts — making it easy for a model to read and summarise.

Prerequisite: set `TAVILY_API_KEY=tvly-...` in your `.env` file. Get a free key at https://tavily.com.

---
## Section 1 — Without Web Search (The Baseline)

In [10]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Loads OPENAI_API_KEY and TAVILY_API_KEY from .env.
# Both are needed for this notebook.

from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# ============================================================
# CELL 2: Create a Basic Agent (No Tools)
# ============================================================
# A plain agent with no tools can only answer from its training
# data. We use this as our baseline to clearly see the difference
# once we add web search in Section 2.

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano"
    # No tools= parameter → agent can only use its internal knowledge
)

In [12]:
# ============================================================
# CELL 3: Ask About the Knowledge Cutoff
# ============================================================
# Asking the model directly about its own knowledge cutoff is a
# good way to understand its limitations before building on top of it.
# The model will typically tell you a date after which it knows nothing.
# Any question about events AFTER that date risks hallucination.

from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [13]:
# ============================================================
# CELL 4: See the Model's Honest Answer About Its Limits
# ============================================================
# The model will tell you its training cutoff date.
# Remember this — anything after that date needs a tool like web search.

print(response['messages'][-1].content)

Short answer: my training data goes up to around November 2023. I don’t have real-time browsing by default, so I may not know events or developments that happened after that date unless browsing is enabled or you provide current information.

If you need up-to-date specifics, tell me the topic and I can either:

- Look up current information if browsing is available to me, or
- Help you form search queries and verify sources on your end.

For anything not time-sensitive, I can usually help quite well with the knowledge I was trained on.


---
## Section 2 — Adding Web Search

### How Tavily fits into the tool pattern

Tavily is just a Python client that makes HTTP requests to a search API. We wrap it in our `@tool` decorator to give it the metadata (name + description) that the model needs to decide when to use it.

The tool returns a **dict** from Tavily containing:
- `results` → list of search results, each with `title`, `url`, `content` (clean excerpt)
- `answer` → Tavily's own AI-generated summary (optional)

The agent reads this dict and uses it to compose a grounded, cited answer.

### Why Tavily over plain Google?

| | Tavily | Raw Google/Bing |
|---|---|---|
| **Returns** | Clean JSON with text excerpts | Raw HTML requiring parsing |
| **LLM-friendly** | Yes — designed for agents | No — messy, tokeniser-heavy |
| **Rate limits** | Generous free tier | Strict / expensive |
| **Setup** | One API key | OAuth + custom parsing |

In [14]:
# ============================================================
# CELL 5: Define the Web Search Tool
# ============================================================
# TavilyClient() reads TAVILY_API_KEY from the environment
# (loaded by load_dotenv() above) — no explicit key needed.
#
# The tool signature:
#   query: str  → the search query the model will construct
#   returns Dict[str, Any] → the full Tavily response dict
#
# The model will:
#   1. Read the user's question
#   2. Formulate a good search query (e.g. "San Francisco mayor 2025")
#   3. Call this tool with that query
#   4. Read the returned results
#   5. Summarise into a natural language answer
#
# The last line tests the tool directly (no agent) — always do this
# to confirm your API key works and the tool returns sane results
# before involving the agent.

from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# Initialise the Tavily client — picks up TAVILY_API_KEY from env
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

# Smoke test — run this before attaching to an agent
web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco',
   'content': 'There have been 45 individuals who have served as mayor in San Francisco since 1850, when California became a state following the American Conquest of California. Prior to the conquest, Californios served as Mayor of San Francisco during the Spanish and Mexican eras since 1779.\n\nThe current mayor is Democrat Daniel Lurie.\n\n## Elections\n\n[edit]\n\nMain articles: 1979 San Francisco mayoral election, 1995, 1999, 2003, 2007, 2011, 2015, 2018, 2019, and 2024 [...] | Image 5 Seal of San Francisco |\n| Image 6 Flag of San Francisco |\n| Image 7.jpg) Incumbent Daniel Lurie since January 8, 2025 |\n| Government of San Francisco |\n| Style "Style (form of address)") | His Honor Mr. Mayor |\n| Residence | No official residence |\n| Term length |  

In [15]:
# ============================================================
# CELL 6: Create a Web-Search-Enabled Agent and Run a Query
# ============================================================
# Now the agent has web_search in its toolkit.
#
# When we ask about the current mayor, the agent will:
#   1. Recognise this is a current-events question (knowledge may be stale)
#   2. Decide to call web_search with a relevant query
#   3. Read the Tavily results
#   4. Compose an answer grounded in today's web data
#
# Compare this answer to what a no-tools agent would give.
# The web-search answer will be accurate; the no-tools answer
# might be outdated or hallucinated.

from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search]   # Give the agent internet access
)

# question = HumanMessage(content="Hey, know the weather in HYD, if its less than 20 degrees, get the flights in between HYD and USA")

question = HumanMessage(content="Hey, get the flights in between HYD and USA as well as HYD and Singapore today")

response = agent.invoke(
    {"messages": [question]}
)

In [16]:
# ============================================================
# CELL 7: Inspect the Full Message Trace
# ============================================================
# Printing all messages shows the agent's full reasoning chain:
#
#   messages[0]  HumanMessage
#       "Who is the current mayor of San Francisco?"
#
#   messages[1]  AIMessage  (tool call decision)
#       content: ""  (empty — model is calling a tool, not replying)
#       tool_calls: [{name: "web_search",
#                    args: {query: "San Francisco mayor 2025"}}]
#       ↑ Notice the model formulated a good search query on its own!
#
#   messages[2]  ToolMessage
#       content: {results: [{title: ..., url: ..., content: ...}, ...]}
#       This is the raw Tavily response — JSON dict from our tool.
#
#   messages[3]  AIMessage  (final answer)
#       content: "The current mayor of San Francisco is..."
#       This is where the model reads the search results and
#       composes a clean, grounded answer for the user.
#
# This trace is invaluable for debugging:
#   - Did the agent search at all?
#   - Was the search query sensible?
#   - Did Tavily return useful results?
#   - Did the model correctly interpret those results?

from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Hey, get the flights in between HYD and USA as well as HYD and Singapore today', additional_kwargs={}, response_metadata={}, id='92939c27-f0c5-4301-84d2-db995c7ac417'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 62, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5d5b340d02', 'id': 'chatcmpl-EUZ8qKc0sa7FNo163jD2Era2rVKiA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fd27-abbe-7071-8a61-727005518708-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'flights f

---
### LangSmith Trace

LangSmith is Anthropic/LangChain's observability platform. A trace captures the full agent run — every LLM call, every tool call, token counts, latency — in a visual UI.

Example trace for this notebook: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r

To enable tracing in your own runs, add to your `.env`:
```
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__...
LANGCHAIN_PROJECT=my-project-name
```
Traces are automatically sent to LangSmith after that — no code changes needed.

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Knowledge cutoff** | All LLMs have a training date — anything after it requires a tool |
| **Tavily** | A search API purpose-built for LLM agents — returns clean JSON, not HTML |
| **`@tool` wrapping** | Turns `tavily_client.search()` into a LangChain tool in 4 lines |
| **Smoke test first** | Always call `web_search.invoke(...)` directly before using in an agent |
| **Message trace** | `messages[1].tool_calls` shows the query the model chose; `messages[2]` shows the raw results |
| **LangSmith** | Set 3 env vars to get free visual tracing of every agent run |

### Next steps
- **1.3** — Memory: making agents remember across multiple turns
- **1.5** — Personal Chef: combining web search + memory into a real application